# Notebook to Compare Heuristics

In [91]:
import asyncio
import nest_asyncio

import random
import pandas as pd
from time import time

import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities

In [92]:
random.seed(42)  # For accurate comparison
nest_asyncio.apply()

In [93]:
from src import (
    kempe_greedy,
    welfare_greedy,
    c_fim,
)

from src import (
    estimate_influence,
    independent_cascade_community,
)

In [94]:
from src import Loader
from pathlib import Path

path_to_networks = Path('../../data/synthetic/networks')
path_to_results = Path('../../results/barbasi_albert/size_200/cfim_welfare')

file_name = 'barbasi_albert_200'

In [95]:
async def main():
    loader = Loader(max_workers=4)
    graph = await loader.load(f'{path_to_networks}/{file_name}.pkl')

    communities = list(greedy_modularity_communities(graph))
    costs = nx.get_node_attributes(graph, 'node_costs')

    return graph, communities, costs


graph, communities, costs = asyncio.run(main())

## Comparison of Kempe Greedy and Welfare Greedy

In [96]:
k = 10  # number of seeds to select
alphas = [1, 0.5, 0, -1, -3, -5, -7, -9]  # inequality-aversion parameter
p = 0.05  # edge activation probability
num_sims = 1000

In [97]:
# results = []
#
# for alpha in alphas:
#     start = time()
#     kempe_seeds = kempe_greedy(
#         graph=graph,
#         k=k,
#         probability=p,
#         num_simulations=num_sims,
#     )
#     kempe_time = time() - start
#
#     kempe_influence = estimate_influence(
#         graph=graph,
#         seeds=kempe_seeds,
#         propagation_prob=p,
#         num_simulations=num_sims,
#     )
#
#     kempe_by_comm = independent_cascade_community(
#         graph=graph,
#         seeds=kempe_seeds,
#         probability=p,
#         num_sims=num_sims // 2,
#     )
#     kempe_by_comm_rounded = {k: round(v, 2) for k, v in kempe_by_comm.items()}
#
#     start = time()
#     welfare_seeds = welfare_greedy(
#         graph=graph,
#         communities=communities,
#         k=k,
#         alpha=alpha,
#         probability=p,
#         num_sims=num_sims,
#     )
#     welfare_time = time() - start
#
#     welfare_influence = estimate_influence(
#         graph=graph,
#         seeds=welfare_seeds,
#         propagation_prob=p,
#         num_simulations=num_sims,
#     )
#
#     welfare_by_comm = independent_cascade_community(
#         graph=graph,
#         seeds=welfare_seeds,
#         probability=p,
#         num_sims=num_sims // 2,
#     )
#     welfare_by_comm_rounded = {k: round(v, 2) for k, v in welfare_by_comm.items()}
#
#     results.append(
#         {
#             'alpha': alpha,
#             'kempe_seeds': kempe_seeds,
#             'kempe_time_s': kempe_time,
#             'kempe_avg_influence': kempe_influence,
#             'kempe_influence_by_community': kempe_by_comm_rounded,
#             'welfare_seeds': welfare_seeds,
#             'welfare_time_s': welfare_time,
#             'welfare_avg_influence': welfare_influence,
#             'welfare_influence_by_community': welfare_by_comm_rounded,
#             'org_total_influence': kempe_influence,
#             'fair_total_influence': welfare_influence,
#         }
#     )
#
# df = pd.DataFrame(results)

## Comparison of Welfare Greedy and Budgeted Welfare Greedy (CFIM)

In [98]:
k = 10  # number of seeds to select
alphas = [1, 0.5, 0, -1, -3, -5, -7, -9]  # inequality-aversion parameter
p = 0.05  # edge activation probability
budget = 5.0  # Total budget available
num_sims = 1000

In [99]:
results = []

for alpha in alphas:
    start = time()
    welfare_seeds = welfare_greedy(
        graph=graph,
        communities=communities,
        k=k,
        alpha=alpha,
        probability=p,
        num_sims=num_sims,
    )
    welfare_time = time() - start

    welfare_influence = estimate_influence(
        graph=graph,
        seeds=welfare_seeds,
        propagation_prob=p,
        num_simulations=num_sims,
    )

    welfare_by_comm = independent_cascade_community(
        graph=graph,
        seeds=welfare_seeds,
        probability=p,
        num_sims=num_sims // 2,
    )
    welfare_by_comm_rounded = {k: round(v, 2) for k, v in welfare_by_comm.items()}

    start = time()
    cfim_seeds = c_fim(
        graph=graph,
        communities=communities,
        max_seeds=k,
        budget=budget,
        costs=costs,
        alpha=alpha,
        probability=p,
    )
    cfim_time = time() - start

    cfim_influence = estimate_influence(
        graph=graph,
        seeds=cfim_seeds,
        propagation_prob=p,
        num_simulations=num_sims,
    )

    cfim_by_comm = independent_cascade_community(
        graph=graph,
        seeds=cfim_seeds,
        probability=p,
        num_sims=num_sims // 2,
    )
    cfim_by_comm_rounded = {k: round(v, 2) for k, v in cfim_by_comm.items()}

    results.append(
        {
            'alpha': alpha,
            'cfim_seeds': cfim_seeds,
            'cfim_time_s': cfim_time,
            'cfim_influence_by_community': cfim_by_comm_rounded,
            'welfare_seeds': welfare_seeds,
            'welfare_time_s': welfare_time,
            'welfare_influence_by_community': welfare_by_comm_rounded,
            'org_total_influence': cfim_influence,
            'fair_total_influence': welfare_influence,
        }
    )

df = pd.DataFrame(results)
df.to_csv(f'{path_to_results}/welfare_cfim_{file_name}_{k}_results.csv', index=False)

No more affordable candidates:  50%|█████     | 5/10 [00:01<00:01,  3.44it/s, seeds=5, cost=5.00/5.00, remaining_budget=0.00, influenced={0: 0.15, 1: 0.08, 2: 0.13, 3: 0.07, 4: 0.17, 5: 0.08, 6: 0.16, 7: 0.16}]
